In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import List, Optional
from datetime import datetime, timedelta

# Input Models

class SafetyStatus(Enum):
    SAFE = "SAFE"
    BLOCKED = "BLOCKED"

@dataclass(frozen=True)
class SafetyStatusRecord:
    """Module 4 input."""
    status: SafetyStatus
    reason_codes: List[str]

@dataclass(frozen=True)
class OvercoolingAssessmentRecord:
    """Module 6 input."""
    overcooling_score: float
    reduction_feasibility_hint: bool

@dataclass(frozen=True)
class CostContextRecord:
    """Module 5 input."""
    tariff_status: str  # Tariff status
    cost_severity_index: int

# Output Model

@dataclass(frozen=True)
class CoolingRecommendationRecord:
    """Optimization recommendation."""
    recommended_reduction_percentage: float
    time_window: str  # Format
    safety_status: str
    explanation: str

# Module Implementation

class EnerviaCoolingOptimizer:
    """Cooling Optimizer."""

    def __init__(self):
        # Constraints
        self.MAX_REDUCTION_CAP = 15.0  # Max reduction
        self.BASE_WINDOW_HOURS = 2

    def generate_cooling_recommendation(
        self, 
        safety: SafetyStatusRecord, 
        overcooling: OvercoolingAssessmentRecord, 
        current_power: float, 
        cost: CostContextRecord
    ) -> CoolingRecommendationRecord:
        """Generate recommendation."""
        now = datetime.now()
        window_str = f"{now.strftime('%H:%M')}–{(now + timedelta(hours=self.BASE_WINDOW_HOURS)).strftime('%H:%M')}"

        # Dependency Check 
        if safety.status == SafetyStatus.BLOCKED:
            return CoolingRecommendationRecord(
                recommended_reduction_percentage=0.0,
                time_window=window_str,
                safety_status="BLOCKED",
                explanation=f"Optimization blocked by Safety Engine. Reason: {', '.join(safety.reason_codes)}"
            )

        # Feasibility Check
        if not overcooling.reduction_feasibility_hint:
            return CoolingRecommendationRecord(
                recommended_reduction_percentage=0.0,
                time_window=window_str,
                safety_status="SAFE",
                explanation="No reduction recommended: System is currently at optimal thermal stability."
            )

        # Calculate Reduction
        base_reduction = overcooling.overcooling_score * 10.0  # Scale
        economic_weight = cost.cost_severity_index / 10.0      # Scale
        
        calculated_percentage = base_reduction * economic_weight
        
        # Apply caps
        final_percentage = min(calculated_percentage, self.MAX_REDUCTION_CAP)

        # Final generation
        return CoolingRecommendationRecord(
            recommended_reduction_percentage=round(final_percentage, 2),
            time_window=window_str,
            safety_status="SAFE",
            explanation=(
                f"Recommended reduction of {final_percentage:.1f}% based on "
                f"overcooling score of {overcooling.overcooling_score} "
                f"during {cost.tariff_status} window (Severity: {cost.cost_severity_index})."
            )

# optimizer = EnerviaCoolingOptimizer()
# recommendation = optimizer.generate_cooling_recommendation(safety_rec, overcooling_rec, 250.5, cost_rec)